# CS336 Spring 2026 - 14 Data Loading / Training Loop / Checkpoint / Sampling
----
> 13 有了 loss、AdamW 与学习率概念，现在把**可复现的最小训练流程**连接起来。

```text
tokenized 1D stream → (inputs, next-token targets)
→ decoder-only LM → logits → cross entropy
→ backward → grad clipping → AdamW
→ evaluation / checkpoint / generation
```

为便于独立运行，本章使用**极小的合成 token stream**，并用 PyTorch 内置 attention 组装轻量 decoder。它演示训练系统而非替代 12 的从零 RoPE/RMSNorm 模型；实际 Assignment 时应换成你自己的 12 对应实现。


## 1. Language Modeling Data Loading

假设训练数据已经 tokenize 成一个长的一维数组：

```text
x0, x1, x2, x3, x4, ...
```

从一个起点 `s` 截出 `T+1` 个 ID：

```text
inputs : x_s,   ... x_(s+T-1)
targets: x_(s+1), ... x_(s+T)
```

所以返回形状是 `(B,T)` 和 `(B,T)`。合法起点必须保证最后一个 target 不越界。


In [1]:
import math
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)


def get_batch(stream: torch.Tensor, batch_size: int, context_len: int, generator=None):
    if stream.ndim != 1 or stream.dtype != torch.long:
        raise ValueError("stream must be a 1D torch.long Tensor")
    if batch_size < 1 or context_len < 1 or len(stream) <= context_len:
        raise ValueError("invalid batch/context size")
    starts = torch.randint(
        low=0,
        high=len(stream) - context_len,
        size=(batch_size,),
        generator=generator,
    )
    offsets = torch.arange(context_len + 1)
    windows = stream[starts[:, None] + offsets[None, :]]
    return windows[:, :-1], windows[:, 1:]


stream = torch.tensor([0, 1, 2, 3] * 80, dtype=torch.long)
gen = torch.Generator().manual_seed(2026)
inputs, targets = get_batch(stream, batch_size=3, context_len=8, generator=gen)
assert inputs.shape == targets.shape == (3, 8)
assert torch.equal(inputs[:, 1:], targets[:, :-1])
print("inputs[0] :", inputs[0].tolist())
print("targets[0]:", targets[0].tolist())


inputs[0] : [1, 2, 3, 0, 1, 2, 3, 0]
targets[0]: [2, 3, 0, 1, 2, 3, 0, 1]


## 2. Data loader 的边界案例

当长度恰好等于 `context_len + 1` 时，只有一个合法窗口；当长度不超过 `context_len` 时无法提供最后一个 target，应该明确拒绝。

正式训练集往往更大，此时使用 numpy `memmap` 或磁盘 token IDs 文件，可以避免全部数据常驻内存；本节只测试索引逻辑。


In [2]:
tiny = torch.arange(9, dtype=torch.long)
xx, yy = get_batch(tiny, batch_size=2, context_len=8)
assert xx.shape == yy.shape == (2, 8)
assert torch.equal(xx[0], torch.arange(8))
assert torch.equal(yy[0], torch.arange(1, 9))

for invalid in (torch.arange(8, dtype=torch.long), torch.arange(0, dtype=torch.long)):
    try:
        get_batch(invalid, batch_size=1, context_len=8)
        raise AssertionError("insufficient stream length should fail")
    except ValueError:
        pass
print("唯一合法窗口 / 数据过短：验证通过")


唯一合法窗口 / 数据过短：验证通过


## 3. 最小训练用 decoder

为专注 Training Loop，本章的模型用 PyTorch 的 `nn.MultiheadAttention`、`LayerNorm` 和简单 FFN。它仍然是带 causal mask、next-token logits `(B,T,V)` 的 decoder。

12 的 RoPE + RMSNorm + SwiGLU 将在正式 `.py` 实现阶段取代这个小模型；本章的 `get_batch`、optimizer、checkpoint、sampler 结构无需因此改变。


In [3]:
class TinyTrainingLM(nn.Module):
    def __init__(self, vocab_size=4, d_model=24, n_heads=4, max_context=8):
        super().__init__()
        self.max_context = max_context
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_context, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 2 * d_model), nn.SiLU(), nn.Linear(2 * d_model, d_model)
        )
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, ids):
        b, t = ids.shape
        if t < 1 or t > self.max_context:
            raise ValueError("invalid context length")
        positions = torch.arange(t, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        n = self.norm1(x)
        mask = torch.ones(t, t, dtype=torch.bool, device=ids.device).triu(1)
        attended, _ = self.attn(n, n, n, attn_mask=mask, need_weights=False)
        x = x + attended
        x = x + self.ffn(self.norm2(x))
        return self.head(x)


model = TinyTrainingLM()
assert model(inputs).shape == (3, 8, 4)
print("tiny causal LM parameters:", sum(p.numel() for p in model.parameters()))


tiny causal LM parameters: 5260


## 4. 一个真正执行的 Training Loop

每个迭代：

1. `optimizer.zero_grad(set_to_none=True)`
2. 构造 `(B,T)` 输入和标签
3. `loss = cross_entropy(logits, labels)`
4. `loss.backward()`
5. `clip_grad_norm_`
6. `optimizer.step()`

训练 token stream 是重复的 `0,1,2,3`，因此 next-token prediction 具有清晰的规律；如果训练过程正确，固定评估 batch 的 loss 应明显下降。


In [4]:
train_stream = torch.tensor([0, 1, 2, 3] * 280, dtype=torch.long)
eval_generator = torch.Generator().manual_seed(888)
eval_x, eval_y = get_batch(
    train_stream, batch_size=16, context_len=8, generator=eval_generator
)


@torch.no_grad()
def evaluate_loss(model, x, y):
    model.eval()
    logits = model(x)
    return float(F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1)))


initial_loss = evaluate_loss(model, eval_x, eval_y)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.008, weight_decay=0.01)
data_generator = torch.Generator().manual_seed(2026)
history = []

model.train()
for step in range(45):
    x, y = get_batch(
        train_stream, batch_size=16, context_len=8, generator=data_generator
    )
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, 4), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    history.append(float(loss.detach()))

final_loss = evaluate_loss(model, eval_x, eval_y)
print("initial evaluation loss:", round(initial_loss, 4))
print("final   evaluation loss:", round(final_loss, 4))
assert final_loss < initial_loss


initial evaluation loss: 1.3041
final   evaluation loss: 0.0


## 5. Train 与 Eval 要分开

`model.train()` 可能启用 dropout；`model.eval()` 切换到推理行为。评估时还应使用 `torch.no_grad()`，避免多余的 autograd graph。

验证集应与训练集**在数据层面独立划分**；上面的固定 batch 只是合成模式上的功能测试，不是模型泛化能力证明。


In [5]:
model.eval()
with torch.no_grad():
    check_logits = model(eval_x)
    check_loss = F.cross_entropy(check_logits.reshape(-1, 4), eval_y.reshape(-1))
assert torch.isfinite(check_loss)
print("evaluation mean NLL:", float(check_loss))
print("evaluation perplexity:", math.exp(float(check_loss)))


evaluation mean NLL: 2.018138047787943e-06
evaluation perplexity: 1.0000020181400842


## 6. Checkpoint 不只是保存模型权重

如果希望从某一步继续训练，需要至少保存：

- `model.state_dict()`
- `optimizer.state_dict()`（尤其 AdamW 的动量）
- 已完成 step 或当前 step
- 影响可复现性的随机数状态，包括 PyTorch RNG 与批次采样 generator

只保存模型参数可以用于 inference，但不足以**严格恢复完整训练状态**。


In [6]:
with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = Path(tmpdir) / "tiny_checkpoint.pt"
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": 45,
        "torch_rng": torch.random.get_rng_state(),
        "data_rng": data_generator.get_state(),
    }
    torch.save(checkpoint, ckpt_path)

    restored_model = TinyTrainingLM()
    restored_optimizer = torch.optim.AdamW(
        restored_model.parameters(), lr=0.008, weight_decay=0.01
    )
    restored = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    restored_model.load_state_dict(restored["model"])
    restored_optimizer.load_state_dict(restored["optimizer"])
    restored_model.eval()

    with torch.no_grad():
        torch.testing.assert_close(model(eval_x), restored_model(eval_x))
    assert restored["step"] == 45
    assert len(restored_optimizer.state) == len(optimizer.state)
    assert torch.equal(restored["data_rng"], data_generator.get_state())
    print("checkpoint/model/optimizer/RNG round-trip：通过")


checkpoint/model/optimizer/RNG round-trip：通过


## 7. Autoregressive Generation：每次只生成一个 token

给定 prompt IDs，取模型最后一个时间位置的 logits：

$$
p_i=\operatorname{softmax}(z_i/\tau).
$$

`temperature` 低时更确定，高时更随机。`top_k` 只保留 k 个候选 logits，其余设为 `-inf`，再 softmax 和采样。

生成时，context 超过模型最大长度，必须截断；本章没有 KV Cache，因此每次都会重新计算当前上下文。


In [7]:
@torch.no_grad()
def sample_next_token(logits, temperature=1.0, top_k=None, generator=None):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    scores = logits / temperature
    if top_k is not None:
        if not (1 <= top_k <= scores.shape[-1]):
            raise ValueError("top_k outside vocabulary range")
        cutoff = scores.topk(top_k, dim=-1).values[..., -1, None]
        scores = scores.masked_fill(scores < cutoff, float("-inf"))
    probs = torch.softmax(scores, dim=-1)
    return torch.multinomial(probs, num_samples=1, generator=generator)


@torch.no_grad()
def generate(model, prompt, num_new_tokens, temperature=1.0, top_k=None, seed=336):
    model.eval()
    ids = prompt.clone()
    generator = torch.Generator(device=ids.device).manual_seed(seed)
    for _ in range(num_new_tokens):
        context = ids[:, -model.max_context :]
        logits = model(context)[:, -1, :]
        nxt = sample_next_token(logits, temperature, top_k, generator)
        ids = torch.cat([ids, nxt], dim=-1)
    return ids


seed_prompt = torch.tensor([[0, 1]], dtype=torch.long)
generated = generate(model, seed_prompt, 12, temperature=0.8, top_k=2)
assert generated.shape == (1, 14)
assert torch.equal(generated[:, :2], seed_prompt)
assert (generated >= 0).all() and (generated < 4).all()
print("generated IDs:", generated[0].tolist())


generated IDs: [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1]


## 8. Sampling 的边界情况

- `num_new_tokens=0`：应直接返回输入；
- `top_k=1`：只剩最高分候选，相当于 greedy 选择（除非并列分数）；
- `temperature<=0`：本实现明确拒绝；
- 空 prompt：没有“最后一个 logits”，应由外层明确规定 BOS/start-token 规则。

生成质量需要真实语料和较大模型。本章只验证 sampling 机制和概率归一化。


In [8]:
empty_generation = generate(model, seed_prompt, 0)
assert torch.equal(empty_generation, seed_prompt)

fixed_logits = torch.tensor([[0.1, 2.0, -3.0, 1.0]])
one = sample_next_token(fixed_logits, top_k=1)
assert one.item() == 1

try:
    sample_next_token(fixed_logits, temperature=0)
    raise AssertionError("invalid temperature should fail")
except ValueError:
    pass
print("zero generation / top-1 / invalid temperature：通过")


zero generation / top-1 / invalid temperature：通过


## 9. 从教学小例子走向 TinyStories

正式训练之前，还需要：

```text
TinyStories / OpenWebText text
    ↓ trained tokenizer.encode
token IDs → efficient storage (e.g. numpy memmap)
    ↓ streaming/seeded random batches
Transformer LM (12) + training system (13,14)
    ↓ checkpoints / validation NLL / samples
```

此外要记录随机种子、学习率、batch size、context length、参数量、tokens/sec 与设备信息，才便于复现和诊断结果。
